(sec:T3:SF:interpretacion)=
### Interpretación espectral de la serie de Fourier

Desde un punto de vista espectral, la serie de Fourier proporciona una representación discreta del contenido en frecuencia de una señal periódica. Cada coeficiente $c_k$ está asociado a una componente armónica de frecuencia $k\omega_0$, de modo que la periodicidad en el dominio temporal se traduce en una discretización del espectro en el dominio de la frecuencia. Como ejemplo, se muestra en la Figura~\ref{fig:SF}, la serie de Fourier correspondiente a un tren de pulsos triangulares.

\begin{figure}[h]
	\centering
	\includegraphics[width=\linewidth]{figs/SF}
	\caption{Serie de Fourier correspondiente a un tren de pulsos triangulares.}
	\label{fig:SF}
\end{figure}

Esta interpretación permite analizar de forma cualitativa el comportamiento de las señales y los sistemas:
- Señales cuyos coeficientes son significativos únicamente para valores pequeños de $|k|$ presentan una variación lenta en el tiempo.
- La presencia de armónicos de orden elevado se asocia a variaciones rápidas o a la existencia de discontinuidades.

Por otro lado, teniendo en cuenta que las exponenciales complejas son autofunciones de los sistemas LTI, el desarrollo en serie de Fourier a la salida de un sistema LTI tendrá como coeficientes:
\begin{equation}
	d_k = H(k\omega_0)\,c_k,
\end{equation}
donde $H(k\omega_0)$ es la respuesta en frecuencia del sistema a las frecuencias armónicas de la fundamental.

Asimismo, esta representación discreta del espectro constituye el punto de partida para establecer relaciones estructurales con otros conceptos del análisis de Fourier, como la transformada de Fourier de señales aperiódicas y la interpretación del muestreo como fenómeno dual de la periodicidad.

In [ ]:
import numpy as np
from bokeh.plotting import figure, show
from bokeh.layouts import column, row
from bokeh.models import ColumnDataSource, CustomJS, Slider, Div, Label
from bokeh.io import output_notebook

# Importamos las funciones auxiliares
from utils.plot_helpers import style_math_axes, add_math_ticks

output_notebook()

# ==========================================
# 1. PARÁMETROS INICIALES
# ==========================================
# Definimos un periodo T0 = 2 (frecuencia pi)
VAL_STEEPNESS = 0.5  # Valor inicial (Bajo = Suave/Senoidal)
N_POINTS = 500       # Resolución temporal para la integración numérica
N_HARMONICS = 25     # Cuántos armónicos calculamos

t_min, t_max = -3.0, 3.0
t_vals = np.linspace(t_min, t_max, N_POINTS)

# ==========================================
# 2. GENERACIÓN DE DATOS
# ==========================================

# A. Datos Tiempo: x(t) = tanh( s * sin(pi * t) ) / tanh(s)
# Normalizamos dividiendo por tanh(s) para que la amplitud siempre sea 1
def generate_signal(t_arr, s):
    # Evitamos s=0
    if s < 1e-3: s = 1e-3
    raw = np.tanh(s * np.sin(np.pi * t_arr))
    norm_factor = np.tanh(s)
    return raw / norm_factor

y_vals = generate_signal(t_vals, VAL_STEEPNESS)
source_time = ColumnDataSource(data=dict(t=t_vals, y=y_vals))

# B. Datos Frecuencia (Cálculo numérico inicial en Python)
# Hacemos una DFT rápida sobre un solo periodo [-1, 1]
t_period = np.linspace(-1, 1, N_POINTS, endpoint=False) # Un periodo exacto
k_vals = np.arange(0, N_HARMONICS + 1) # Solo positivos para magnitud (simetría par en modulo)

def calc_dft(s):
    if s < 1e-3: s = 1e-3
    y_p = np.tanh(s * np.sin(np.pi * t_period)) / np.tanh(s)
    
    mags = []
    # Calculamos ck = (1/N) * sum( x[n] * exp(-j*2*pi*k*n/N) )
    # Al ser integral continua aproximada: ck = (1/T0) * integral
    # Usamos la aproximacion discreta normalizada
    for k in k_vals:
        # Proyectamos sobre la base exponencial compleja
        # Como la señal es impar real, ck será imaginario puro.
        # Solo nos interesa la magnitud.
        # Integracion trapecial simple o suma de Riemann
        # ck = (1/T) * int(x(t) * e^-jwkt dt)
        # Aproximación: dot product
        basis_real = np.cos(k * np.pi * t_period)
        basis_imag = -np.sin(k * np.pi * t_period)
        
        re = np.mean(y_p * basis_real)
        im = np.mean(y_p * basis_imag)
        
        mag = np.sqrt(re**2 + im**2)
        # Ajuste para serie de Fourier exponencial (doble lado) vs unilateral
        # El calculo da la magnitud correcta de ck
        mags.append(mag)
        
    return np.array(mags)

mag_vals = calc_dft(VAL_STEEPNESS)
# Filtramos valores muy pequeños para limpieza visual
mag_vals[mag_vals < 1e-4] = 0

source_freq = ColumnDataSource(data=dict(k=k_vals, mag=mag_vals, zeros=np.zeros_like(k_vals)))


# ==========================================
# 3. GRÁFICOS
# ==========================================

# --- TIEMPO ---
p_time = figure(width=600, height=300, title="Señal en el Tiempo: x(t)")
style_math_axes(p_time, x_range=(t_min, t_max), y_range=(-1.3, 1.3), xlabel="t", ylabel="x(t)")
add_math_ticks(p_time, yticks=[-1, 1], ytick_labels=["-1", "1"], tick_len=5)

p_time.line('t', 'y', source=source_time, color="#1f77b4", line_width=3, legend_label="x(t)")
p_time.legend.location = "top_right"

# --- FRECUENCIA ---
p_freq = figure(width=600, height=300, title="Espectro de Magnitud: |ck|")
style_math_axes(p_freq, x_range=(-1, N_HARMONICS+1), y_range=(0, 0.7), xlabel="|k|", ylabel="|ck|")

# Stems
p_freq.segment(x0='k', y0='zeros', x1='k', y1='mag', source=source_freq, color="#d62728", line_width=3)
p_freq.scatter('k', 'mag', source=source_freq, color="#d62728", size=8, marker="circle", legend_label="|ck|")
p_freq.legend.location = "top_right"


# ==========================================
# 4. INTERACTIVIDAD
# ==========================================
# Slider logarítmico simulado o rango amplio para ver el efecto
s_steep = Slider(start=0.1, end=10.0, value=VAL_STEEPNESS, step=0.1, title=r"Abrupticidad (Pendiente)")

callback = CustomJS(
    args=dict(source_t=source_time, source_f=source_freq, s_steep=s_steep),
    code="""
    const s = s_steep.value;
    const PI = Math.PI;
    
    // 1. GENERAR DATOS TIEMPO
    const t = source_t.data['t'];
    const y = source_t.data['y'];
    const N_t = t.length;
    
    // Normalización: tanh(s)
    let norm = Math.tanh(s);
    if (Math.abs(norm) < 1e-9) norm = 1.0;

    for (let i = 0; i < N_t; i++) {
        y[i] = Math.tanh(s * Math.sin(PI * t[i])) / norm;
    }
    source_t.change.emit();

    // 2. CALCULAR DFT (Fourier) EN TIEMPO REAL
    // Usamos un solo periodo para integrar: t va de 0 a 2 (T0=2) o de -1 a 1.
    // Generamos internamente un vector de un periodo con resolución fina
    const N_integ = 200; // Puntos para integrar
    const k_arr = source_f.data['k'];
    const mag = source_f.data['mag'];
    
    // Pre-calculamos señal en un periodo [-1, 1]
    let y_p = new Float32Array(N_integ);
    let t_p = new Float32Array(N_integ);
    for(let i=0; i<N_integ; i++){
        // t va de -1 a 1 (excluyendo el ultimo punto para periodicidad estricta)
        let ti = -1 + (2 * i / N_integ);
        t_p[i] = ti;
        y_p[i] = Math.tanh(s * Math.sin(PI * ti)) / norm;
    }
    
    // Bucle para cada armónico k
    for (let i = 0; i < k_arr.length; i++) {
        let k = k_arr[i];
        let sum_re = 0.0;
        let sum_im = 0.0;
        
        // Integración numérica (Suma de Riemann)
        for (let j = 0; j < N_integ; j++) {
            // e^(-j * omega0 * k * t) -> omega0 = pi
            let angle = PI * k * t_p[j];
            let cos_val = Math.cos(angle);
            let sin_val = Math.sin(angle); // -sin para exp complejo negativo
            
            sum_re += y_p[j] * cos_val;
            sum_im += y_p[j] * (-sin_val);
        }
        // Dividir por N_integ (promedio, equivalente a 1/T integral)
        // Ojo: Para la serie exponencial compleja ck
        // ck = (1/T0) * integral. Aquí estamos promediando muestras, es correcto.
        let re = sum_re / N_integ;
        let im = sum_im / N_integ;
        
        let m = Math.sqrt(re*re + im*im);
        
        // Limpieza de ruido numérico
        if (m < 1e-3) m = 0;
        mag[i] = m;
    }
    source_f.change.emit();
""")

s_steep.js_on_change('value', callback)

# ==========================================
# 5. LAYOUT Y CAPTION
# ==========================================
caption_text = """
<div style="font-family: sans-serif; margin-top: 15px; font-size: 14px; color: #444;">
    <p><b>Interpretación Tiempo-Frecuencia:</b></p>
    <ul>
        <li><b>Señal Suave (Abrupticidad baja $\approx 0.5$):</b> La señal se asemeja a una senoide pura. En frecuencia, la energía se concentra casi exclusivamente en el armónico fundamental ($k=1$). Coeficientes de orden alto nulos.</li>
        <li><b>Señal Abrupta (Abrupticidad alta $> 5$):</b> La señal se satura y aparecen cambios bruscos (discontinuidades de salto), pareciéndose a una onda cuadrada. Esto obliga a que aparezcan <b>armónicos de orden elevado</b> ($k=3, 5, 7...$) con amplitudes significativas.</li>
    </ul>
    <p style="font-size:13px; opacity:0.8;"><i>Nota: Una variación rápida en el tiempo siempre implica contenido en altas frecuencias.</i></p>
</div>
"""
caption = Div(text=caption_text)

layout = column(s_steep, p_time, p_freq, caption, sizing_mode="scale_width")
show(layout)